# Instalación de dependencias

Antes de iniciar la generación de datos sintéticos se instalan las librerías necesarias para garantizar que el entorno de ejecución cuente con todas las dependencias requeridas por el proyecto.

## Instalación de `dbldatagen`

```python
!pip install dbldatagen
```

**Justificación**

Se instala `dbldatagen` porque será la librería principal para generar datos sintéticos sobre Apache Spark. Permite crear grandes volúmenes de información de manera distribuida, definir esquemas, generar identificadores únicos y configurar reglas de generación para cada columna.

---

## Instalación de `jmespath`

```python
!pip install jmespath
```

**Justificación**

`jmespath` es una dependencia utilizada para realizar consultas sobre estructuras JSON y es requerida por algunas librerías del proyecto. Su instalación evita errores de dependencias durante la ejecución y facilita el procesamiento de configuraciones estructuradas cuando sea necesario.

---

## Instalación de `pyspark`

```python
!pip install pyspark
```

**Justificación**

Se instala Apache Spark para disponer del motor de procesamiento distribuido sobre el cual se generarán y transformarán los datos sintéticos. PySpark proporciona los DataFrames, funciones SQL y tipos de datos utilizados durante todo el desarrollo.

---

## Instalación de `Faker`

```python
!pip install Faker
```

**Justificación**

`Faker` permite generar información ficticia con apariencia realista, como nombres, empresas, ciudades, direcciones y otros atributos descriptivos. Su uso incrementa la variedad de los datos sintéticos y hace que los conjuntos de datos sean más cercanos a escenarios reales.

---

## Reinicio del intérprete de Python

```python
dbutils.library.restartPython()
```

**Justificación**

Después de instalar nuevas dependencias es recomendable reiniciar el intérprete de Python para que todas las librerías queden correctamente cargadas en la sesión activa. Esto evita problemas de importación y asegura que el notebook utilice las versiones recién instaladas.

> **Nota:** Este reinicio finaliza la sesión actual de Python, por lo que las variables definidas previamente deberán ejecutarse nuevamente en las siguientes celdas.

In [0]:
!pip install dbldatagen
!pip install jmespath
!pip install pyspark
!pip install Faker

dbutils.library.restartPython()

# Importación de librerías

En esta sección se importan las librerías necesarias para la generación de datos sintéticos utilizando Apache Spark y `dbldatagen`.

## `dbldatagen`

```python
import dbldatagen as dg
```

**Justificación**

Se utiliza `dbldatagen` como motor principal para la generación de datos sintéticos. Esta librería permite crear grandes volúmenes de información de forma distribuida sobre Spark, manteniendo el esquema de las tablas y facilitando la definición de reglas como:

- Rangos numéricos.
- Valores aleatorios.
- Listas de valores.
- Distribuciones de datos.
- Generación de identificadores únicos.

Su integración con Spark permite generar conjuntos de datos escalables sin necesidad de construir manualmente cada DataFrame.

---

## Funciones de Spark SQL

```python
from pyspark.sql import functions as F
```

**Justificación**

Se importan las funciones de Spark SQL para realizar transformaciones posteriores sobre los DataFrames generados.

Estas funciones permiten:

- Crear columnas calculadas.
- Manipular fechas.
- Generar valores aleatorios.
- Aplicar expresiones SQL.
- Limpiar y transformar datos antes de almacenarlos.

Se utiliza el alias `F` porque es el estándar de desarrollo en proyectos con PySpark, mejorando la legibilidad del código.

---

## Tipos de datos de Spark

```python
from pyspark.sql.types import (
    IntegerType,
    FloatType,
    StringType,
    TimestampType,
    StructField,
    BooleanType,
    StructType,
    ArrayType,
    DecimalType
)
```

**Justificación**

Se importan los tipos de datos de Spark para definir explícitamente la estructura de cada columna durante la generación de información sintética.

Definir el tipo de dato evita conversiones implícitas, mejora la calidad del esquema y garantiza que los datos generados sean compatibles con formatos como Parquet y Delta Lake.

Los tipos utilizados son:

| Tipo | Justificación |
|-------|---------------|
| `IntegerType` | Identificadores, cantidades y códigos numéricos. |
| `FloatType` | Valores decimales aproximados como pesos o porcentajes. |
| `DecimalType` | Valores monetarios con precisión controlada. |
| `StringType` | Información descriptiva como nombres, categorías o ciudades. |
| `BooleanType` | Estados lógicos como activo/inactivo o verdadero/falso. |
| `TimestampType` | Fechas y horas de eventos o transacciones. |
| `StructType` | Definición de estructuras complejas o registros anidados. |
| `StructField` | Definición individual de los campos dentro de una estructura. |
| `ArrayType` | Almacenamiento de colecciones de elementos en una misma columna. |

---

## Resultado esperado

Con estas librerías se dispone de todos los componentes necesarios para construir los DataFrames sintéticos del proyecto RetailMax, manteniendo una estructura tipada, consistente y preparada para su posterior procesamiento en Apache Spark.

In [0]:
import dbldatagen as dg
from pyspark.sql import functions as F
from pyspark.sql.types import (
    IntegerType, FloatType, StringType, TimestampType, 
    StructField, BooleanType, StructType, ArrayType, DecimalType
)

In [0]:
import yaml

# 1. Definir la ruta de tu archivo físico
ruta_archivo = '/Workspace/Users/jose.dataengineer@hotmail.com/Retailmax_data/data-generation/config.yaml'


try:
    # 2. Abrir y leer el archivo físico
    # Usamos encoding='utf-8' por buenas prácticas, especialmente al manejar español
    with open(ruta_archivo, 'r', encoding='utf-8') as archivo:
        datos = yaml.safe_load(archivo)
        
    # 3. Extraer solo las llaves de primer nivel
    llaves_principales = list(datos.keys())
    print(f"Llaves de primer nivel: {llaves_principales}")
    
    # 4. Extraer las llaves de los elementos internos
    llaves_internas_tiendas = list(datos['tipos_tienda'][0].keys())
    print(f"Llaves internas de las tiendas: {llaves_internas_tiendas}")

except FileNotFoundError:
    print(f"Error: No se encontró el archivo '{ruta_archivo}'. Verifica que esté en la misma carpeta.")
except yaml.YAMLError as e:
    print(f"Error al analizar el archivo YAML: {e}")

In [0]:
import yaml
import dbldatagen as dg
from pyspark.sql.types import StringType


# Cargar YAML
with open('/Workspace/Users/jose.dataengineer@hotmail.com/Retailmax_data/data-generation/config.yaml', 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)



# Aplicar al generador

generador_articulos = (
        dg.DataGenerator(spark, name="MSTR_ARTICULOS", rows=5000)
        .withIdOutput() # Generar automáticamente IDs para la columna "art_id"
        .withColumn("cod_barra", StringType(), template=r"###########")
        #.withColumn("desc_art", StringType(), values=nombres_productos)
        .withColumn("id_categ_n1", IntegerType(), minValue=1, maxValue=10)
        .withColumn("id_categ_n2", IntegerType(), minValue=1, maxValue=40)
        .withColumn("id_categ_n3", IntegerType(), minValue=1, maxValue=150)
        .withColumn("id_proveedor", IntegerType(), minValue=1, maxValue=800)
        .withColumn("precio_lista", FloatType(), minValue=1000, maxValue=250000)
        .withColumn("peso_kg", FloatType(), minValue=0.1, maxValue=25)
        .withColumn("unid_medida", StringType(), values=["UND", "KG", "LT", "CJ"])
        .withColumn("activo", BooleanType())
        #.withColumn("fecha_alta", DateType(), begin="2025-01-01", end="2026-01-01")
    )

df_articulos = generador_articulos.build()
display(df_articulos)

In [0]:
generador_proveedores = (
    dg.DataGenerator(spark, name="MSTR_PROVEEDORES", rows=800)
    .withIdOutput()
    .withColumn("razon_social", StringType(), values=proveedores)
    .withColumn("pais_origen", StringType(), values=paises)
    .withColumn("tiempo_repo_dias", IntegerType(), minValue=1, maxValue=45)
    .withColumn("calificacion_calidad", FloatType(), minValue=1, maxValue=5)
    .withColumn("activo", BooleanType())
)

df_proveedores = generador_proveedores.build()
display(df_proveedores)

In [0]:
generador_clientes = (
    dg.DataGenerator(spark, name="CRM_MIEMBROS", rows=50000)
    .withIdOutput() # "id_miembro"
    .withColumn("id_ciudad", IntegerType(), minValue=1, maxValue=40)
    .withColumn("genero", StringType(), values=["M", "F", "Otro"])
    .withColumn("rango_edad", StringType(), values=["18-25", "26-35", "36-45", "46-60", "60+"])
    .withColumn("canal_pref", StringType(), values=["Web", "App", "Tienda"])
    .withColumn("activo", BooleanType())
    .withColumn("fec_registro", DateType(), begin="2018-01-01", end="2026-01-01")
)

In [0]:
generador_ventas = (
    dg.DataGenerator(spark, name="TRANS_VENTAS", rows=1000000)
    .withIdOutput()# "id_venta"
    .withColumn("id_miembro", IntegerType(), minValue=1, maxValue=50000)
    .withColumn("id_tienda", IntegerType(), minValue=1, maxValue=150)
    .withColumn("art_id", IntegerType(), minValue=1, maxValue=5000)
    .withColumn("qty_vendida", IntegerType(), minValue=1, maxValue=20)
    .withColumn("precio_unitario_venta", FloatType(), minValue=1000, maxValue=250000)
    .withColumn("descuento_aplicado", FloatType(), minValue=0, maxValue=0.40)
)

In [0]:
generador_stock = (
    dg.DataGenerator(spark, name="INV_STOCK_DIARIO", rows=750000)
    .withIdOutput() # "id_snapshot"
    .withColumn("art_id", IntegerType(), minValue=1, maxValue=5000)
    .withColumn("id_tienda", IntegerType(), minValue=1, maxValue=150)
    .withColumn("stock_fisico", IntegerType(), minValue=0, maxValue=1000)
    .withColumn("stock_transito", IntegerType(), minValue=0, maxValue=200)
    .withColumn("stock_reservado", IntegerType(), minValue=0, maxValue=100)
)

In [0]:
generador_devoluciones = (
    dg.DataGenerator(spark, name="POST_DEVOLUCIONES", rows=50000)
    .withIdOutput() # "id_devolucion"
    .withColumn("id_trans_origen", IntegerType(), minValue=1, maxValue=1000000)
    .withColumn("art_id", IntegerType(), minValue=1, maxValue=5000)
    .withColumn("id_tienda", IntegerType(), minValue=1, maxValue=150)
    .withColumn("qty_devuelta", IntegerType(), minValue=1, maxValue=5)
    .withColumn("motivo_cod", StringType(), values=["DEFECTO", "GARANTIA", "CLIENTE", "VENCIDO"])
    .withColumn("estado_devolucion", StringType(), values=["Pendiente", "Aprobada", "Rechazada"])
)

In [0]:
# Build the DataFrame
df = data_spec.build()

# Explore the Generated Data
display(df)